* Context chaining
    * 복잡한 다단계 처리를 하나의 프롬프트가 아니라 더 단순하고 집중된 프롬프트 여러 개를 이용해 작업을 처리하는 방식

* Branching structure(분기 구조)
    * 컨텍스트 연쇄 구조의 위력을 보여 줌 
    * 분기 구조에 의해 단일 경로 작업 흐름이 병렬로 진행되는 다중 작업 분석으로 변화

In [59]:
import os 
import yaml
from pathlib import Path
from openai import OpenAI

In [60]:
CONFIG_PATH = Path("config.yaml")

with CONFIG_PATH.open("r") as f:
    cfg = yaml.safe_load(f)

### Layer 1: Establishing the scope (The 'What')

1. Set the Signal-to-Noise Ratio
    * Cleaning the data
    * Seperate substantive content from conversational noise

In [61]:
# 회의록
meeting_transcript = """
        Tom: Morning all. Coffee is still kicking in.
        Sarah: Morning, Tom. Right, let's jump in. Project Phoenix timeline. Tom, you said the backend components are on track?
        Tom: Mostly. We hit a small snag with the payment gateway integration. It's... more complex than the docs suggested. We might need another three days.
        Maria: Three days? Tom, that's going to push the final testing phase right up against the launch deadline. We don't have that buffer.
        Sarah: I agree with Maria. What's the alternative, Tom?
        Tom: I suppose I could work over the weekend to catch up. I'd rather not, but I can see the bind we're in.
        Sarah: Appreciate that, Tom. Let's tentatively agree on that. Maria, what about the front-end?
        Maria: We're good. In fact, we're a bit ahead. We have some extra bandwidth.
        Sarah: Excellent. Okay, one last thing. The marketing team wants to do a big social media push on launch day. Thoughts?
        Tom: Seems standard.
        Maria: I think that's a mistake. A big push on day one will swamp our servers if there are any initial bugs. We should do a soft launch, invite-only for the first week, and then do the big push. More controlled.
        Sarah: That's a very good point, Maria. A much safer strategy. Let's go with that. Okay, great meeting. I'll send out a summary.
        Tom: Sounds good. Now, more coffee.
"""

In [62]:
# prompt g2: 실질적인 내용 분리
prompt_g2 = f"""
        Analyze the following meeting transcript. 
        Your task is to isolate the substantive content from the conversational noise.
        - Substantive content includes: decisions made, project updates, problems raised, and strategic suggestions.
        - Noise includes: greetings, pleasantries, and off-topic remarks (like coffee).
        Return ONLY the substantive content.

        Transcript:
        ---
        {meeting_transcript}
        ---
"""

In [63]:
try:
    client = OpenAI(
        base_url=cfg["llm"]["base_url"],        #local host
        api_key=cfg["llm"]["api_key"]           #local key
    )

    response_g2 = client.chat.completions.create(
        model=cfg["llm"]["model"],
        messages=[
            {"role":"system", "content": cfg["llm"]["system_prompt"]},
            {"role":"user", "content": prompt_g2}
        ],
        max_tokens=cfg["llm"]["max_tokens"],
        temperature=cfg["llm"]["temperature"]
    )
    
    substantive_content = response_g2.choices[0].message.content
    print("--- Substantive Content ---")
    print(substantive_content)

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Connection error.


2. Define the Scope of Time(Simulated RAG)
    * previous summary 가지고 오기
    * teaches the user the importance of historical context

In [64]:
previous_summary = "In our last meeting, we finalized the goals for Project Phoenix and assigned backend work to Tom and front-end to Maria."

In [65]:
prompt_g3 = f"""
    Context: The summary of our last meeting was: "{previous_summary}"
    Task: Analyze the following substantive content from our new meeting.
    Identify and summarize ONLY the new developments, problems, or decisions that have occurred since the last meeting. 

    New Meeting Content:
    ---
    {substantive_content}
    ---
"""

In [66]:
try:
    response_g3 = client.chat.completions.create(
        model=cfg["llm"]["model"],
        messages=[
            {"role":"system", "content": cfg["llm"]["system_prompt"]},
            {"role":"user", "content": prompt_g3}
        ],
        max_tokens=cfg["llm"]["max_tokens"],
        temperature=cfg["llm"]["temperature"]
    )

    new_developments = response_g3.choices[0].message.content
    print("--- New Developments ---")
    print(new_developments)

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Connection error.


### Layer 2: Conducting the Investigation(The 'How')
* generating insights
* the core of the semantic context interpreation

1. Identify the Key Threads
    * Ask the AI to read between the lines.
    * 침묵, 망설임 등의 암시적인 신호 추출

In [67]:
# Uncovering Implicit Treads
prompt_g4 = f"""
    Task: Analyze the following meeting content for implicit social dynamics and unstated feelings. 
    Go beyond the literal words. 
    - Did anyone seem hesitant or reluctant despite agreeing to something?
    - Were there any underlying disagreements or tensions?
    - What was the overall mood?

    Meeting Content: 
    ---
    {substantive_content}
    ---
"""

In [68]:
try:
    response_g4 = client.chat.completions.create(
        model=cfg["llm"]["model"],
        messages=[
            {"role":"system", "content": cfg["llm"]["system_prompt"]},
            {"role":"user", "content": prompt_g4}
        ],
        max_tokens=cfg["llm"]["max_tokens"],
        temperature=cfg["llm"]["temperature"]
    )

    implicit_threads = response_g4.choices[0].message.content
    print("--- Implicit Threads ---")
    print(implicit_threads)

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Connection error.


2. Perform "Intellectual Combinations"
    *   

* solve a problem by synthesizing differenct ideas

In [69]:
prompt_g5 = f"""
    Context: In the meeting, 
    Maria suggested a 'soft launch' to avoid server strain, 
    and also mentioned her team has 'extra bandwidth'.
    
    Tom is facing a 3-day delay on the backend.

    Task: Propose a novel, actionable idea 
    hat uses Maria's team's extra bandwidth to help mitigate Tom's 3-day delay. 
    Combine these two separate pieces of information into a single solution.
"""

In [70]:
try:
    response_g5 = client.chat.completions.create(
        model=cfg["llm"]["model"],
        messages=[
            {"role":"system", "content": cfg["llm"]["system_prompt"]},
            {"role":"user", "content": prompt_g5}
        ],
        max_tokens=cfg["llm"]["max_tokens"],
        temperature=cfg["llm"]["temperature"]
    )

    novel_solution = response_g5.choices[0].message.content
    print("--- Novel Solution ---")
    print(novel_solution)

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Connection error.


### Layer3: Determining the Action (The 'Waht Next')
* concrete , forward looking artifacts

1. Define the Output Format (Final Summary)

In [71]:
prompt_g6 = f"""
    Task: Create a final, concise summaryof the meeting in a markdown table.
    Use the following information to construct the table. 

    New Developments: {new_developments}

    The table should have three columns: "Topic", "Decision/Outcome", and "Owner".
    ---
"""

In [72]:
try:
    response_g6 = client.chat.completions.create(
        model=cfg["llm"]["model"],
        messages=[
            {"role":"system", "content": cfg["llm"]["system_prompt"]},
            {"role":"user", "content": prompt_g6}
        ],
        max_tokens=cfg["llm"]["max_tokens"],
        temperature=cfg["llm"]["temperature"]
    )

    summary_table = response_g6.choices[0].message.content
    print("--- Summary Table ---")
    print(summary_table)

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Connection error.


2. Generate the Subsequent Task

In [73]:
prompt_g7 = f"""
    Task: Based on the following summary table, 
    draft a polite and professional follow-up email to the team (Sarah, Tom, Maria).
    The email should clearly state the decisions made and the action items for each person.

    Summary Table: 
    ---
    {summary_table}
    ---
"""

In [74]:
try: 
    response_g7 = client.chat.completions.create(
        model=cfg["llm"]["model"],
        messages=[
            {"role":"system", "content": cfg["llm"]["system_prompt"]},
            {"role":"user", "content": prompt_g7}
        ],
    )
    follow_up_email = response_g7.choices[0].message.content
    print("--- Follow-up Email ---")
    print(follow_up_email)

except Exception as e: 
    print(f"An error occurred: {e}")

An error occurred: Connection error.
